# Actor-Critic Methods

In the previous chapter we learned a policy directly. The policy was updated by gradient ascent on the expected return, and the variance of the gradient estimate was reduced by subtracting a learned baseline from the Monte Carlo return.

$\displaystyle V(S_t;\phi)$

That baseline was useful, but it had a modest role: it centred the returns, and nothing more. The training signal of the policy was still the full return,

$\displaystyle G_t$

with all its noise, and the agent still had to wait until the end of an episode before learning anything.

In this chapter the value network becomes more active. Instead of only comparing the observed return with a reference value, we use the value estimate to replace part of the return. After one or a few real rewards, the rest of the episode is summarised by a bootstrap estimate,

$\displaystyle V(S_{t+n};\phi)$

This is the same idea that took us from Monte Carlo prediction to temporal-difference learning in Chapter 05, now applied to policy gradients. Once the value network is used inside the policy-update target, it is no longer merely a baseline: it becomes a **critic**. The policy network is the **actor**. The critic evaluates the actor's actions, and the actor changes the policy using the critic's evaluation.

This brings us back to the pattern of **generalised policy iteration** from Chapter 03: evaluation and improvement, interleaved. The critic evaluates the current policy; the actor improves it; the critic then has to follow the policy that has just changed.

The chapter proceeds in four steps. First we move from a baseline to a bootstrapping critic, and make the bias-variance trade-off explicit. Then we implement the simplest one-step actor-critic algorithm. Next we introduce the two ideas behind **A3C** and **A2C**: multi-step returns and parallel actors. Finally we introduce **Generalised Advantage Estimation**, which replaces the sharp choice of a rollout length with a smoother bias-variance knob, and we apply the resulting method to Cart-Pole and Lunar Lander.

## From Baseline to Critic

The policy update of VPG weights the score of each action by the sample advantage,

$\displaystyle A(S_t, A_t) = G_t - V(S_t;\phi)$

Here the return

$\displaystyle G_t$

is an unbiased sample of the action value,

$\displaystyle q_\pi(S_t, A_t)$

Chapter 05 taught us that the return is not the only possible estimate of a value. We can bootstrap: stop after one step and let the value function estimate the rest,

$\displaystyle R_{t+1} + \gamma\, V(S_{t+1};\phi)$

If we use this one-step estimate in place of the full return, the advantage estimate becomes

$\displaystyle A(S_t, A_t) = R_{t+1} + \gamma\, V(S_{t+1};\phi) - V(S_t;\phi) = \delta_t$

The last quantity is the familiar **TD error**. This is the central bridge of the chapter: the TD error is both a prediction error for the critic and, under an exact critic, an advantage estimate for the actor.

To see this, suppose the critic is exact,

$\displaystyle V = v_\pi$

Then the conditional expectation of the TD error is

$\displaystyle \mathbb{E}_\pi\big[ \delta_t \mid S_t = s, A_t = a \big] = \mathbb{E}\big[ R_{t+1} + \gamma\, v_\pi(S_{t+1}) \mid S_t = s, A_t = a \big] - v_\pi(s)$

By the Bellman relation between state values and action values, this is

$\displaystyle q_\pi(s,a) - v_\pi(s) = a_\pi(s,a)$

So, with an exact critic, the TD error is an unbiased estimate of the advantage. This observation is the foundation of actor-critic methods. The critic does not need to estimate action values explicitly: a state-value function is enough, because the TD error already measures whether the chosen action turned out better or worse than expected from that state.

### Bias and Variance

Why prefer the TD error to the Monte Carlo advantage? The answer is the same bias-variance trade-off we met in Chapter 05, but now the estimate is used to update a policy.

A Monte Carlo update uses the complete future return and subtracts the value estimate as a baseline:

$\displaystyle G_t - V(S_t)$

The baseline may be imperfect, so this expression is not necessarily an exact sample of the true advantage. But because the baseline depends on the state and not on the action, it does not bias the policy-gradient estimate. Its variance is high, because every future reward depends on future states, future actions, and environment randomness.

A one-step advantage uses a single reward and then bootstraps:

$\displaystyle R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$

Its variance is much lower, because most of the future is replaced by one number. Its bias comes from the critic: if the estimate of the next state is wrong, the actor is pushed in a systematically wrong direction.

An n-step advantage sits between these two extremes:

$\displaystyle G_{t:t+n} - V(S_t)$

It uses n real rewards before bootstrapping. More real rewards mean more variance, but the bootstrap value is discounted by

$\displaystyle \gamma^n$

so the critic's error has less influence than in the one-step target.

There are two practical consequences. First, the agent can learn before the episode ends, and even in continuing tasks that do not naturally end. Second, the target now depends on the current critic parameters,

$\displaystyle \phi$

so we are back to the moving-target problem of Chapter 07. The critic must be trained with a **semi-gradient** update: the bootstrapped target is treated as a constant while differentiating the loss.

As in Chapter 07, we also distinguish termination from truncation. If an episode genuinely terminates, there is no future value. If it is stopped by a time limit, the last state still has a future value and we should bootstrap from it. We use a terminal flag that is one only for genuine termination,

$\displaystyle d_{t+1}$

and write the one-step critic target as

$\displaystyle y_t = R_{t+1} + \gamma\, (1 - d_{t+1})\, V(S_{t+1};\phi)$

### Actor and Critic

We now have two coupled learning processes.

The **critic** learns to evaluate the current policy with a state-value function,

$\displaystyle V(s;\phi)$

For one-step actor-critic, it minimises the squared TD error,

$\displaystyle \mathcal{L}_v(\phi) = \big( y_t - V(S_t;\phi) \big)^2$

with the target treated as a constant.

The **actor** learns to improve the policy,

$\displaystyle \pi_\theta(a \mid s)$

It ascends the TD-error-weighted score, with an entropy bonus for exploration,

$\displaystyle \delta_t \log \pi_\theta(A_t \mid S_t) + \beta H\big(\pi_\theta(\cdot \mid S_t)\big)$

During the actor update, the TD error is also treated as a constant. The actor should learn from the critic's evaluation; it should not change the critic through the policy loss.

The coupling goes both ways. The critic's targets depend on actions sampled by the actor, and the actor's updates depend on the critic's estimates. Early in training both are poor, so the actor is improved using imperfect evaluations. This is the same small miracle as generalised policy iteration: if the critic tracks the current policy fast enough and the actor changes slowly enough, the two improve together. In practice the critic's learning rate is often at least as large as the actor's, and the actor's steps must be kept small. We will return to this point in Chapter 11.

![](./images/networks.png)

## One-Step Actor-Critic

The simplest actor-critic algorithm follows directly from the previous section: at every time step, compute the TD error and update both networks. It is the policy gradient counterpart of TD(0), and like TD(0) it is fully **online**: there is no need to store a trajectory. We start again from Cart-Pole:

In [ ]:
import gymnasium

cart_pole = gymnasium.make('CartPole-v1', render_mode="rgb_array")

print('State Space: ', cart_pole.observation_space.shape)
print('Action Space: ', cart_pole.action_space.n)

In [ ]:
import torch
import numpy as np
import random

# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

cart_pole.reset(seed=seed)
cart_pole.action_space.seed(seed);

The actor and the critic are the policy and value networks of Chapter 08, unchanged:

In [ ]:
def create_policy_network(env, first_hidden_layer=128, second_hidden_layer=64):

    # size of the state vector and number of actions
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    # a fully connected network with two hidden layers
    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: one preference (logit) per action
        torch.nn.Linear(second_hidden_layer, action_size),
    )

    return dnn


def create_value_network(env, first_hidden_layer=128, second_hidden_layer=64):

    # size of the state vector
    state_size = env.observation_space.shape[0]

    # a fully connected network with two hidden layers
    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: the estimated value of the state
        torch.nn.Linear(second_hidden_layer, 1),
    )

    return dnn


pi = create_policy_network(cart_pole)
v = create_value_network(cart_pole)

So are the functions that define the action distribution, sample an action (returning its log-probability and the entropy of the distribution), and the two evaluation policies, stochastic and greedy. The only difference is that the distribution function now accepts a **batch of states** as well as a single state; this costs nothing here and will be needed later in the chapter:

In [ ]:
def policy_distribution(states):

    # convert one state, or a batch of states, into a tensor
    states = torch.as_tensor(np.asarray(states), dtype=torch.float32)

    # the softmax over the preferences defines the action distribution
    return torch.distributions.Categorical(logits=pi(states))


def select_action(state):

    # sample an action from the policy
    dist = policy_distribution(state)
    action = dist.sample()

    # log-probability of the sampled action and entropy of the distribution
    return int(action.item()), dist.log_prob(action), dist.entropy()


def stochastic_pi(state):

    # sample an action from the policy, as during training
    return select_action(state)[0]


def greedy_pi(state):

    # select the most probable action
    with torch.no_grad():
        preferences = pi(torch.as_tensor(state, dtype=torch.float32))

    return int(torch.argmax(preferences).item())

In [ ]:
def evaluate(env, policy, episodes=1):

    # collect total rewards per episode
    rewards = []

    for episode in range(episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        total_reward = 0.0

        # run one episode
        while not done:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

        rewards.append(total_reward)

    # return the average reward over episodes
    return float(np.mean(rewards))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def show_policy(env, policy, n_episodes=1, max_steps=500):

    for episode in range(n_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        step = 0

        # run the episode until done or max_steps
        while not done and step < max_steps:

            # make a step
            action = policy(state)
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # render the environment
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.axis("off")
            plt.title(f"Episode {episode+1}, Step {step}")
            plt.show()
            plt.close()

            step += 1

Each network has its own optimizer. Unlike Chapter 08, where the two networks shared the same learning rate, we give the critic a **larger learning rate** than the actor: the critic must track a policy that keeps changing, while the actor must move slowly enough for the critic's evaluations to remain meaningful:

In [ ]:
actor_learning_rate = 0.0002
critic_learning_rate = 0.001

policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=actor_learning_rate)
value_optimizer = torch.optim.RMSprop(v.parameters(), lr=critic_learning_rate)

### Loss Functions

The optimization step receives a single transition. It computes the bootstrapped target with the critic, **without tracking gradients**, so that the target is a constant (the semi-gradient update of Chapter 07); the TD error is the difference between the target and the current value estimate. The critic minimises the squared TD error. The actor maximises the TD error-weighted log-probability of the action taken, plus the entropy bonus, with the TD error *detached*, so that the actor's loss does not modify the critic:

In [ ]:
def optimize(state, logpa, entropy, reward, next_state, terminated, gamma=0.99, beta=0.01):

    # current value estimate (with gradient)
    value = v(torch.as_tensor(state, dtype=torch.float32)).squeeze(-1)

    # bootstrapped target: a constant, computed without gradient;
    # a terminal next state has no future value
    with torch.no_grad():
        next_value = v(torch.as_tensor(next_state, dtype=torch.float32)).squeeze(-1)
        target = reward + gamma * (1.0 - float(terminated)) * next_value

    # the TD error is the advantage estimate
    td_error = target - value

    # critic loss: squared TD error (semi-gradient: the target is constant)
    value_loss = td_error.pow(2)

    # actor loss: the TD error is a constant weight on the score
    policy_loss = -(td_error.detach() * logpa + beta * entropy)

    # update the critic
    value_optimizer.zero_grad()
    value_loss.backward()
    value_optimizer.step()

    # update the actor
    policy_optimizer.zero_grad()
    policy_loss.backward()
    policy_optimizer.step()

    return policy_loss.item(), value_loss.item()

### Algorithm

The one-step actor-critic algorithm is the following:

1. Use two neural networks: an actor with one preference per action, and a critic with a single value output.
2. In the current state, sample an action from the actor and take it in the environment.
3. Compute the one-step target with the critic:

$\displaystyle y_t = R_{t+1} + \gamma (1 - d_{t+1}) V(S_{t+1};\phi)$

4. Compute the TD error:

$\displaystyle \delta_t = y_t - V(S_t;\phi)$

5. Update the critic by gradient descent on

$\displaystyle \delta_t^2$

with the target treated as a constant.

6. Update the actor by gradient ascent on

$\displaystyle \delta_t \log \pi_\theta(A_t \mid S_t) + \beta H(\pi_\theta(\cdot \mid S_t))$

with the TD error treated as a constant.

7. Move to the next state; if the episode is over, reset the environment.

Since learning no longer happens only at episode boundaries, we organise the training loop around **environment steps** rather than episodes. We evaluate the greedy policy at regular intervals of steps. This is the standard way of measuring progress in deep reinforcement learning, because it makes algorithms with different update schedules comparable: what matters is how much experience each of them needs. The evaluation runs in a separate copy of the environment, so it does not interrupt the episode in progress:

In [ ]:
def actor_critic(env, max_steps, eval_every=2000, gamma=0.99, beta=0.01):

    # a separate, seeded environment for evaluation
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    state, _ = env.reset()

    for step in range(1, max_steps + 1):

        # act in the environment
        action, logpa, entropy = select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)

        # one optimization step on this transition
        optimize(state, logpa, entropy, reward, next_state, terminated, gamma, beta)

        # move on; reset the environment at the end of the episode
        state = next_state
        if terminated or truncated:
            state, _ = env.reset()

        # evaluate the greedy policy at regular intervals
        if step % eval_every == 0:
            score = evaluate(eval_env, greedy_pi, episodes=10)
            scores.append(score)
            print(f"Step {step:07}, score {score:05.1f}", end="\r", flush=True)

    return scores

### Solve Cart-Pole

We train the agent for a fixed budget of environment steps:

In [ ]:
scores = actor_critic(cart_pole, max_steps=150_000)

In [ ]:
agent_score = evaluate(cart_pole, greedy_pi, episodes=10)
print("Agent score", agent_score)

In [ ]:
show_policy(cart_pole, greedy_pi, n_episodes=1, max_steps=500)

To study the learning process we adapt the experiment function of the previous chapters to the step-based training loop. The only differences are that the training budget is a number of steps, and that the results record the evaluation interval, so that the learning curves can be plotted against the number of environment steps:

In [ ]:
import joblib
from pathlib import Path

def experiment(
    env,
    train,
    setup,
    max_steps,
    name,
    eval_every=2000,
    seeds=(12, 34, 56, 78, 90),
    smoothing_window=5,
):

    results_path = Path("temp") / f"{name}_results.joblib"

    # load saved results if they already exist
    if results_path.exists():
        experiment_results = joblib.load(results_path)

        n_points = len(experiment_results["mean_score"])
        final_mean = experiment_results["mean_score"][-1]
        final_min = experiment_results["min_score"][-1]
        final_max = experiment_results["max_score"][-1]

        print(f"Loaded saved results from {results_path}")
        print(f"Smoothed points: {n_points}")
        print(f"Smoothing window: {experiment_results['smoothing_window']}")
        print(f"Final score: mean={final_mean:.1f}, range=[{final_min:.1f}, {final_max:.1f}]")

        return experiment_results

    results = []

    for seed in seeds:
        print(f"Experiment seed: {seed}")

        # set seeds for reproducibility
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        env.reset(seed=seed)
        env.action_space.seed(seed)

        # initialize networks and optimizers
        setup()

        # train the agent
        scores = train(env, max_steps=max_steps, eval_every=eval_every)

        # smooth the learning curve
        kernel = np.ones(smoothing_window) / smoothing_window
        scores = np.convolve(scores, kernel, mode="valid")

        results.append(scores)
        print()

    results = np.array(results)

    experiment_results = {
        "max_score": np.max(results, axis=0),
        "min_score": np.min(results, axis=0),
        "mean_score": np.mean(results, axis=0),
        "smoothing_window": smoothing_window,
        "eval_every": eval_every,
        "seeds": seeds,
        "max_steps": max_steps,
    }

    results_path.parent.mkdir(exist_ok=True)
    joblib.dump(experiment_results, results_path)

    return experiment_results

In [ ]:
# define the experiment setup and hyperparameters
gamma = 0.99                  # discount factor
beta = 0.01                   # entropy coefficient
actor_learning_rate = 0.0002  # step size of the actor's optimizer
critic_learning_rate = 0.001  # step size of the critic's optimizer
first_hidden_layer = 128      # size of the first hidden layer
second_hidden_layer = 64      # size of the second hidden layer

def setup_actor_critic():
    global pi, v, policy_optimizer, value_optimizer
    pi = create_policy_network(cart_pole, first_hidden_layer, second_hidden_layer)
    v = create_value_network(cart_pole, first_hidden_layer, second_hidden_layer)
    policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=actor_learning_rate)
    value_optimizer = torch.optim.RMSprop(v.parameters(), lr=critic_learning_rate)

# run the experiment
actor_critic_results = experiment(
    env=cart_pole,
    train=actor_critic,
    setup=setup_actor_critic,
    max_steps=150_000,
    name="actor_critic",
)

The plotting functions are those of the previous chapters, with environment steps on the horizontal axis:

In [ ]:
def steps_axis(results):
    # the smoothed curves start after the first smoothing window
    offset = results.get("smoothing_window", 1)
    return (np.arange(len(results["mean_score"])) + offset) * results.get("eval_every", 1)


def plot_scores(results, title="Score", color="orange"):

    steps = steps_axis(results)

    plt.figure(figsize=(12, 6))
    plt.title(title)
    plt.xlabel("Environment steps")
    plt.ylabel("Score")

    plt.plot(steps, results["mean_score"], color=color, linewidth=2, label="Mean over seeds")
    plt.fill_between(steps, results["min_score"], results["max_score"], color=color, alpha=0.3, label="Range over seeds")

    plt.legend(loc="upper left")
    plt.show()


def plot_comparison(
    first_results,
    second_results,
    first_label="A",
    second_label="B",
    first_color="blue",
    second_color="red",
):

    first_steps = steps_axis(first_results)
    second_steps = steps_axis(second_results)

    plt.figure(figsize=(12, 6))
    plt.title(f"{second_label} vs {first_label}")
    plt.xlabel("Environment steps")
    plt.ylabel("Score")

    # plot the first algorithm
    plt.plot(first_steps, first_results["mean_score"], color=first_color, linewidth=2, label=f"{first_label} mean")
    plt.fill_between(first_steps, first_results["min_score"], first_results["max_score"], color=first_color, alpha=0.20, label=f"{first_label} range")

    # plot the second algorithm
    plt.plot(second_steps, second_results["mean_score"], color=second_color, linewidth=2, label=f"{second_label} mean")
    plt.fill_between(second_steps, second_results["min_score"], second_results["max_score"], color=second_color, alpha=0.20, label=f"{second_label} range")

    plt.legend()
    plt.show()


plot_scores(actor_critic_results, title="One-step actor-critic score")

The one-step actor-critic does learn, and it learns from the very first steps, since it does not wait for episodes to end. Its learning curve, however, is far from smooth: the score can jump from the maximum to that of the random policy between two consecutive evaluations, and the range across seeds is wide. The reasons are instructive, because they motivate everything that follows. First, each update is computed from **a single transition**: it is a stochastic gradient step with a mini-batch of size one, and consecutive transitions are **strongly correlated**, since they come from the same episode. Gradient-based optimizers work best with batches of diverse, roughly independent samples, as we discussed at length in Chapter 07, and here they get the opposite. Second, early in training the critic is poor, and the **one-step target inherits all of its error**: the actor is being pushed around by evaluations that are mostly wrong. In Chapter 07 the first problem was solved with a replay memory, but that option is not available here: the policy gradient must be estimated with data generated by the **current** policy, and a transition stored a thousand updates ago was generated by a different one.

## Asynchronous Advantage Actor-Critic (A3C)

The [**Asynchronous Advantage Actor-Critic**](./papers/2016%20-%20Asynchronous%20Methods%20for%20Deep%20Reinforcement%20Learning.pdf) algorithm, introduced by Mnih and colleagues in 2016, addresses the two weaknesses of the one-step method: one transition is too little for a stable gradient, and consecutive transitions from one environment are too correlated.

### n-Step Returns

The first idea is to update every n steps instead of every step. The agent collects a short segment of rewards and then bootstraps on the critic's estimate of the state it has reached:

$\displaystyle G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^{n}\, (1 - d_{t+n})\, V(S_{t+n};\phi)$

If the episode terminates before the segment reaches n steps, the sum stops at the terminal reward and no bootstrapping occurs. If the episode is truncated, the value of the last state is bootstrapped as usual.

The corresponding advantage estimate and critic target are

$\displaystyle A^{(n)}_t = G_{t:t+n} - V(S_t;\phi), \qquad y_t = G_{t:t+n}$

Each update uses all transitions in the segment, and each transition has its own n-step return. This is the bias-variance compromise from the previous section: n real rewards reduce the dependence on the critic, while the target remains much less noisy than a full Monte Carlo return.

The returns of a whole segment can be computed in a single backward pass, exactly as in Chapter 08. The recursion is

$\displaystyle G_{t:t+n} = R_{t+1} + \gamma\, (1 - d_{t+1})\, G_{t+1:t+n}$

The only difference from Monte Carlo returns is the starting value of the backward recursion: instead of starting from zero, we start from the bootstrap value of the last state,

$\displaystyle V(S_{t+n};\phi)$

At any given moment the workers are in different states and often at different points of different episodes, so the transitions they generate are much less correlated than those of a single agent. Each worker collects an n-step segment with its local networks, computes the gradients of the actor and critic losses, and applies them to a set of shared global networks. It then copies the updated global parameters into its local networks and continues.

The updates are **asynchronous**: workers do not wait for one another, and the shared parameters are modified without locks, in the style of the [**Hogwild!**](./papers/2011%20-%20HOGWILD!-%20A%20Lock-Free%20Approach%20to%20Parallelizing%20Stochastic%20Gradient%20Descent.pdf) algorithm for parallel stochastic gradient descent. Occasionally a worker overwrites part of another worker's update. In exchange, the algorithm avoids the cost of synchronisation.

The algorithm is the following:

1. Create shared actor and critic networks, and several workers, each with its own environment and local copies of the networks.
2. Each worker independently copies the shared parameters into the local networks.
3. Each worker acts for n steps, or until the end of an episode, recording states, actions, rewards, and terminal flags.
4. Each worker computes n-step returns, bootstrapping with the local critic on the last state, and computes the corresponding n-step advantages.
5. Each worker computes gradients of the actor and critic losses with the local networks and applies them to the shared networks asynchronously.
6. The process repeats until the training budget is exhausted.

A3C was a landmark result: it matched or surpassed DQN on the Atari benchmark using only CPUs, in a fraction of the training time, and it showed that parallelism can replace replay memory as a source of decorrelated data.

A faithful implementation is awkward in our setting. True parallelism in Python requires multiple processes, and multiple processes do not cooperate well with an interactive notebook. More importantly, lock-free updates make the algorithm non-deterministic: two runs can differ even with the same seed. For the controlled experiments of the course, we therefore implement the synchronous variant described next. It keeps the two key ideas of A3C, n-step returns and parallel actors, and removes the asynchrony.

## Advantage Actor-Critic (A2C)

The **Advantage Actor-Critic** algorithm, A2C, is the synchronous version of A3C. Instead of many independent worker-learners, there is one learner and many actors. Several copies of the environment are stepped in lockstep, all using the same current policy. After n steps, all collected transitions are used for a single update of the single pair of networks.

If there are K environments and the rollout length is n, each update uses

$\displaystyle K \times n$

transitions. There is no shared memory to protect, no race condition, and the computation is deterministic given the seed. The forward pass of the actor can also be batched across all environments, which is much more efficient than calling the network once per environment. In practice A2C performs about as well as A3C, and it is the version used in many libraries.

The K environments stepped together form a **vectorised environment**: an object that takes a vector of actions and returns a vector of observations, rewards, and flags. Gymnasium provides an optimised implementation, `gymnasium.vector`, but its automatic handling of episode boundaries hides exactly the details we want to inspect. We therefore implement the mechanism by hand with a list of environments. The states of the environments are kept in an array with one row per environment:

In [ ]:
def create_environments(env, n_envs, seed):

    # K independent copies of the environment, each with its own seed
    envs = [gymnasium.make(env.spec.id) for _ in range(n_envs)]

    states = []
    for i, e in enumerate(envs):
        state, _ = e.reset(seed=seed + i)
        e.action_space.seed(seed + i)
        states.append(state)

    return envs, np.array(states, dtype=np.float32)

### Collecting a Rollout

The n steps of the K environments form a **rollout**. At each step, a single batched forward pass of the actor gives the action distribution in all current states. We sample one action per environment and record the log-probabilities, the entropies, and the values predicted by the critic. Each environment is then stepped with its action.

Two boundary cases need care.

When an episode **terminates**, the terminal flag is recorded. The return recursion will stop there, and the environment is reset so that it can continue to produce experience.

When an episode is **truncated** by the time limit, the state reached still has a future value, but the environment is reset and that state will not be the start of a stored transition. We therefore bootstrap immediately: we add the discounted value of the reached state to the last reward,

$\displaystyle \gamma V(S_{t+1};\phi)$

and mark the step as an episode boundary. This gives the correct n-step return while still allowing the environment to reset:

In [ ]:
def collect_rollout(envs, states, n_steps, gamma=0.99):

    n_envs = len(envs)

    # quantities recorded along the rollout, one row per step, one column per environment
    logpas, entropies, values = [], [], []
    rewards = np.zeros((n_steps, n_envs), dtype=np.float32)
    dones = np.zeros((n_steps, n_envs), dtype=np.float32)

    for t in range(n_steps):

        # one batched forward pass of the actor for all the environments
        dist = policy_distribution(states)
        actions = dist.sample()

        logpas.append(dist.log_prob(actions))
        entropies.append(dist.entropy())
        values.append(v(torch.as_tensor(states)).squeeze(-1))

        # step every environment with its own action
        next_states = np.zeros_like(states)

        for i, env in enumerate(envs):
            next_state, reward, terminated, truncated, _ = env.step(int(actions[i]))

            # a truncated episode has a future: bootstrap on the state reached
            if truncated and not terminated:
                with torch.no_grad():
                    reward += gamma * v(torch.as_tensor(next_state, dtype=torch.float32)).item()

            # at the end of an episode, start a new one
            if terminated or truncated:
                next_state, _ = env.reset()

            rewards[t, i] = reward
            dones[t, i] = float(terminated or truncated)
            next_states[i] = next_state

        states = next_states

    # the critic's estimate of the last states, to bootstrap the unfinished episodes
    with torch.no_grad():
        last_values = v(torch.as_tensor(states)).squeeze(-1).numpy()

    rollout = {
        "logpas": torch.stack(logpas),
        "entropies": torch.stack(entropies),
        "values": torch.stack(values),
        "rewards": rewards,
        "dones": dones,
        "last_values": last_values,
    }

    return rollout, states

### n-Step Returns

Given the rewards, the terminal flags, and the critic's estimate of the last states, the n-step returns of the whole rollout are computed in one backward pass. The recursion starts from the bootstrap values and, in each environment, restarts from zero wherever an episode genuinely ended:

In [ ]:
def compute_returns(rewards, dones, last_values, gamma=0.99):

    # G_t = R_{t+1} + gamma * (1 - d_{t+1}) * G_{t+1}, starting from the bootstrap values
    returns = np.zeros_like(rewards)
    future_return = last_values

    for t in reversed(range(len(rewards))):
        future_return = rewards[t] + gamma * (1.0 - dones[t]) * future_return
        returns[t] = future_return

    return torch.as_tensor(returns)

### Loss Functions

The optimization step is VPG applied to the rollout transitions, with n-step returns in place of Monte Carlo returns. The critic regresses its predictions towards the returns, which are constants because the bootstrap values were computed without gradients. The actor maximises the advantage-weighted log-probabilities plus the entropy bonus.

We add one safeguard that is standard in actor-critic implementations: **gradient clipping**. Before the optimizer step, the gradient vector of each network is rescaled if its norm exceeds a threshold. Policy gradient estimates occasionally take very large values, because a single surprising return multiplies the score of every action in the segment. One such update can collapse the policy; clipping limits the damage without changing the direction of ordinary-sized updates.

A second common safeguard, left to the exercises, is to **normalise the advantages** of each batch to zero mean and unit variance. This makes the size of the actor's steps less dependent on the scale of the rewards:

In [ ]:
def optimize(rollout, returns, beta=0.01, max_gradient_norm=0.5):

    # advantages: n-step returns minus predicted values
    advantages = returns - rollout["values"]

    # critic loss: regression of V(s) towards the n-step returns
    value_loss = advantages.pow(2).mean()

    # actor loss: advantages are treated as constants (detached)
    policy_loss = -(advantages.detach() * rollout["logpas"] + beta * rollout["entropies"]).mean()

    # update the critic
    value_optimizer.zero_grad()
    value_loss.backward()
    torch.nn.utils.clip_grad_norm_(v.parameters(), max_gradient_norm)
    value_optimizer.step()

    # update the actor
    policy_optimizer.zero_grad()
    policy_loss.backward()
    torch.nn.utils.clip_grad_norm_(pi.parameters(), max_gradient_norm)
    policy_optimizer.step()

    return policy_loss.item(), value_loss.item()

### Algorithm

The A2C algorithm is the following:

1. Create K copies of the environment and reset them.
2. For n steps, compute the action distribution in all current states with a single batched forward pass, sample one action per environment, step the environments, and record log-probabilities, entropies, values, rewards, and terminal flags.
3. Compute the n-step returns of the rollout, bootstrapping with the critic on the last states.
4. Update the critic towards the returns and the actor along the advantage-weighted scores, with gradient clipping.
5. Repeat from the rollout collection until the budget of environment steps is exhausted.

Each iteration consumes

$\displaystyle K \times n$

environment steps. The original A3C used five-step rollouts with sixteen workers. We use eight environments and rollouts of twenty steps, so one update is made every

$\displaystyle 8 \times 20 = 160$

environment steps. The final bootstrap value is discounted by

$\displaystyle \gamma^{20} \approx 0.82$

at the end of each rollout:

In [ ]:
def a2c(env, max_steps, eval_every=2000, n_envs=8, n_steps=20, gamma=0.99, beta=0.01, eval_policy=None):

    # the policy used for evaluation (greedy by default)
    eval_policy = greedy_pi if eval_policy is None else eval_policy

    # the K environments and a separate, seeded environment for evaluation
    envs, states = create_environments(env, n_envs, seed=int(env.np_random.integers(1_000_000)))
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    total_steps = 0
    next_evaluation = eval_every

    while total_steps < max_steps:

        # collect a rollout of n steps in the K environments
        rollout, states = collect_rollout(envs, states, n_steps, gamma)
        total_steps += n_steps * n_envs

        # n-step returns and one optimization step
        returns = compute_returns(rollout["rewards"], rollout["dones"], rollout["last_values"], gamma)
        optimize(rollout, returns, beta)

        # evaluate the policy at regular intervals
        if total_steps >= next_evaluation:
            score = evaluate(eval_env, eval_policy, episodes=10)
            scores.append(score)
            next_evaluation += eval_every
            print(f"Step {total_steps:07}, score {score:05.1f}", end="\r", flush=True)

    return scores

### Solve Cart-Pole

We train the agent with the same budget of environment steps used for the one-step actor-critic:

In [ ]:
# set the seeds for reproducibility of results
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
cart_pole.reset(seed=seed)

# fresh networks and optimizers
pi = create_policy_network(cart_pole)
v = create_value_network(cart_pole)
policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=actor_learning_rate)
value_optimizer = torch.optim.RMSprop(v.parameters(), lr=critic_learning_rate)

scores = a2c(cart_pole, max_steps=150_000)

In [ ]:
agent_score = evaluate(cart_pole, greedy_pi, episodes=10)
print("Agent score", agent_score)

And we run the multi-seed experiment:

In [ ]:
# define the experiment setup and hyperparameters
gamma = 0.99                  # discount factor
beta = 0.01                   # entropy coefficient
actor_learning_rate = 0.0002  # step size of the actor's optimizer
critic_learning_rate = 0.001  # step size of the critic's optimizer
n_envs = 8                    # number of parallel environments (K)
n_steps = 20                  # steps per rollout (n)
first_hidden_layer = 128      # size of the first hidden layer
second_hidden_layer = 64      # size of the second hidden layer

def setup_a2c():
    global pi, v, policy_optimizer, value_optimizer
    pi = create_policy_network(cart_pole, first_hidden_layer, second_hidden_layer)
    v = create_value_network(cart_pole, first_hidden_layer, second_hidden_layer)
    policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=actor_learning_rate)
    value_optimizer = torch.optim.RMSprop(v.parameters(), lr=critic_learning_rate)

def train_a2c(env, max_steps, eval_every):
    return a2c(env, max_steps, eval_every, n_envs=n_envs, n_steps=n_steps, gamma=gamma, beta=beta)

# run the experiment
a2c_results = experiment(
    env=cart_pole,
    train=train_a2c,
    setup=setup_a2c,
    max_steps=150_000,
    name="a2c",
)

In [ ]:
plot_comparison(
    actor_critic_results,
    a2c_results,
    first_label="One-step actor-critic",
    second_label="A2C",
    first_color="orange",
    second_color="blue",
)

With the same number of environment steps, A2C learns faster and far more reliably than the one-step actor-critic. The mean curve rises earlier and more steadily, and the band between the best and worst seed is much narrower.

The two algorithms use the same networks, the same learning rates, and the same budget of experience. Every transition is used exactly once in both cases. What changed is the **quality of each update**. A2C updates from a mini-batch of

$\displaystyle 160$

transitions, collected from eight different episodes, so the samples are much less correlated. The targets also use twenty-step returns, so they are less dominated by the critic's early errors than one-step targets.

It is worth noting that this improvement comes without replay memory. DQN decorrelates samples by storing old experience and resampling it; A2C obtains a similar effect by generating fresh on-policy experience in parallel.

### Sharing the Network

Most A2C implementations use a **single network** with two heads, a policy head and a value head, on top of a shared trunk, like the dueling architecture of Chapter 07. The two heads are trained with a single combined loss,

$\displaystyle \mathcal{L}(\theta) = \mathcal{L}_\pi(\theta) + c_v\, \mathcal{L}_v(\theta) - \beta\, H$

Here the parameter vector denotes the whole network, trunk and heads together,

$\displaystyle \theta$

The coefficient

$\displaystyle c_v$

usually around

$\displaystyle 0.5$

balances the value loss against the policy loss. This matters because the value loss depends on the scale of the returns, while the policy loss has a different scale.

Sharing is especially valuable when the state is an image: the convolutional feature extractor is by far the most expensive part of the network, and it would be wasteful to train two of them. The price is that the two heads compete for the same features, and the relative scale of the two losses becomes an additional hyperparameter to tune. For the low-dimensional states of this chapter, two separate networks are simpler and perform equally well, so we keep them. The shared architecture is a straightforward modification:

In [ ]:
class ActorCriticNetwork(torch.nn.Module):

    def __init__(self, env, first_hidden_layer=128, second_hidden_layer=64):
        super().__init__()

        state_size = env.observation_space.shape[0]
        action_size = env.action_space.n

        # shared feature extractor
        self.shared = torch.nn.Sequential(
            torch.nn.Linear(state_size, first_hidden_layer),
            torch.nn.ReLU(),
            torch.nn.Linear(first_hidden_layer, second_hidden_layer),
            torch.nn.ReLU(),
        )

        # policy head: one preference per action
        self.policy_head = torch.nn.Linear(second_hidden_layer, action_size)

        # value head: the value of the state
        self.value_head = torch.nn.Linear(second_hidden_layer, 1)

    def forward(self, state):
        features = self.shared(state)
        return self.policy_head(features), self.value_head(features)

## Generalised Advantage Estimation (GAE)

The number of steps before bootstrapping controls the position of the estimate on the bias-variance spectrum, but it is a crude knob: one integer, and a sharp boundary between real rewards and the future summarised by the critic.

In Chapter 05, the same problem for value estimation was solved by the **lambda-return**. It is an exponentially weighted average of all n-step returns. A single parameter,

$\displaystyle \lambda \in [0,1]$

moves continuously from TD(0) to Monte Carlo. [**Generalised Advantage Estimation**](./papers/2016%20-%20High-Dimensional%20Continuous%20Control%20Using%20Generalized%20Advantage%20Estimation.pdf), introduced by Schulman and colleagues in 2016, applies the same idea to the advantage.

### Advantages as Sums of TD Errors

The key observation is that every n-step advantage can be written as a sum of one-step TD errors. Consider the n-step advantage,

$\displaystyle A^{(n)}_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^{n} V(S_{t+n}) - V(S_t)$

and the TD errors,

$\displaystyle \delta_{t+l} = R_{t+l+1} + \gamma V(S_{t+l+1}) - V(S_{t+l})$

Adding and subtracting the intermediate value terms,

$\displaystyle \gamma^{l} V(S_{t+l}) \qquad l = 1, \ldots, n-1$

makes the sum telescope:

$\displaystyle A^{(1)}_t = \delta_t, \qquad A^{(2)}_t = \delta_t + \gamma\, \delta_{t+1}, \qquad A^{(n)}_t = \sum_{l=0}^{n-1} \gamma^{l}\, \delta_{t+l}$

The one-step advantage is a single TD error. The Monte Carlo advantage is the discounted sum of all TD errors in the episode, with the value of the terminal state equal to zero. Every intermediate estimate is a partial sum of the same sequence.

### The Exponentially Weighted Average

Following the construction of the lambda-return, we average the n-step advantages with exponentially decaying weights,

$\displaystyle (1-\lambda)\lambda^{n-1}$

which sum to one. This gives

$\displaystyle A^{\text{GAE}(\gamma,\lambda)}_t = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} A^{(n)}_t$

Substituting the telescoped form and exchanging the order of summation, each TD error appears in all longer n-step advantages. The total weight of the TD error at offset l is

$\displaystyle (1-\lambda)\gamma^{l} \sum_{n=l+1}^{\infty} \lambda^{n-1} = \gamma^{l} \lambda^{l}$

The result is remarkably simple:

$\displaystyle A^{\text{GAE}(\gamma,\lambda)}_t = \sum_{l=0}^{\infty} (\gamma\lambda)^{l}\, \delta_{t+l}$

The generalised advantage estimate is a discounted sum of TD errors, with discount

$\displaystyle \gamma\lambda$

At one extreme,

$\displaystyle \lambda = 0$

it reduces to the one-step TD error: low variance, high bias. At the other extreme,

$\displaystyle \lambda = 1$

it becomes the Monte Carlo advantage: unbiased, but noisy. Intermediate values interpolate between the two. Values around

$\displaystyle \lambda = 0.95$

are typical.

For a finite rollout, GAE is computed in one backward pass with the recursion

$\displaystyle A^{\text{GAE}}_t = \delta_t + \gamma\lambda\, (1 - d_{t+1})\, A^{\text{GAE}}_{t+1}$

The terminal flag stops the sum at genuine episode ends. At the end of the rollout the sum is truncated, and the last TD error uses the bootstrap value of the last state.

The critic still needs a regression target. Since GAE estimates the advantage,

$\displaystyle q_\pi(S_t,A_t) - v_\pi(S_t)$

we add back the current value estimate:

$\displaystyle y_t = A^{\text{GAE}}_t + V(S_t;\phi)$

This is the lambda-return of Chapter 05, written in terms of TD errors.

In [ ]:
def compute_gae(rewards, dones, values, last_values, gamma=0.99, lam=0.95):

    # the critic's predictions, as constants
    values = values.detach().numpy()

    advantages = np.zeros_like(rewards)
    future_advantage = np.zeros_like(last_values)
    next_values = last_values

    for t in reversed(range(len(rewards))):

        # one-step TD error, with no future value after a terminal state
        td_error = rewards[t] + gamma * (1.0 - dones[t]) * next_values - values[t]

        # A_t = delta_t + gamma * lambda * (1 - d_{t+1}) * A_{t+1}
        future_advantage = td_error + gamma * lam * (1.0 - dones[t]) * future_advantage
        advantages[t] = future_advantage

        next_values = values[t]

    # the critic target is the advantage plus the current value (the lambda-return)
    returns = advantages + values

    return torch.as_tensor(returns)

Since the function returns the critic targets, the rest of the algorithm is unchanged: the advantages used by the actor are the targets minus the predicted values, exactly as before. We add the option to the training function:

In [ ]:
def a2c(env, max_steps, eval_every=2000, n_envs=8, n_steps=20, gamma=0.99, beta=0.01, lam=None, eval_policy=None):

    # the policy used for evaluation (greedy by default)
    eval_policy = greedy_pi if eval_policy is None else eval_policy

    # the K environments and a separate, seeded environment for evaluation
    envs, states = create_environments(env, n_envs, seed=int(env.np_random.integers(1_000_000)))
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    total_steps = 0
    next_evaluation = eval_every

    while total_steps < max_steps:

        # collect a rollout of n steps in the K environments
        rollout, states = collect_rollout(envs, states, n_steps, gamma)
        total_steps += n_steps * n_envs

        # critic targets: n-step returns, or generalised advantage estimation
        if lam is None:
            returns = compute_returns(rollout["rewards"], rollout["dones"], rollout["last_values"], gamma)
        else:
            returns = compute_gae(rollout["rewards"], rollout["dones"], rollout["values"], rollout["last_values"], gamma, lam)

        # one optimization step
        optimize(rollout, returns, beta)

        # evaluate the policy at regular intervals
        if total_steps >= next_evaluation:
            score = evaluate(eval_env, eval_policy, episodes=10)
            scores.append(score)
            next_evaluation += eval_every
            print(f"Step {total_steps:07}, score {score:05.1f}", end="\r", flush=True)

    return scores

We compare the two critic targets: the n-step return and the GAE target. We use the same rollout length on Cart-Pole,

$\displaystyle n = 20$

and the standard GAE setting,

$\displaystyle \lambda = 0.95$

Within a rollout, the GAE sum is truncated after at most twenty TD errors. The two targets therefore differ only in how the TD errors inside the rollout are weighted. The n-step return uses powers of the discount alone,

$\displaystyle \gamma^l$

while GAE uses the shorter effective discount,

$\displaystyle (\gamma\lambda)^l$

In [ ]:
lam = 0.95                    # GAE parameter

def train_a2c_gae(env, max_steps, eval_every):
    return a2c(env, max_steps, eval_every, n_envs=n_envs, n_steps=n_steps, gamma=gamma, beta=beta, lam=lam)

a2c_gae_results = experiment(
    env=cart_pole,
    train=train_a2c_gae,
    setup=setup_a2c,
    max_steps=150_000,
    name="a2c_gae",
)

In [ ]:
plot_comparison(
    a2c_results,
    a2c_gae_results,
    first_label="A2C (n = 20)",
    second_label="A2C with GAE (n = 20, λ = 0.95)",
    first_color="blue",
    second_color="green",
)

On Cart-Pole the two variants reach similar final performance and the differences between the curves are small, within the variability across seeds. This is expected: Cart-Pole rewards are deterministic and the critic learns the value function quickly, so there is little variance for GAE to remove and little bias from the bootstrap. The advantage of GAE becomes substantial in tasks with long horizons and noisy rewards, such as the continuous control benchmarks on which it was introduced, and it is for this reason that it has become a fixed component of the algorithms of the following chapters, PPO in particular.

## Solve Lunar Lander

We close the chapter with the Lunar Lander environment of Chapter 08. There, VPG had learned to hover but not to land after

$\displaystyle 3000$

episodes. A2C with GAE requires no change in the code, but the hyperparameters do change, as they usually do when the environment changes.

We use more parallel environments and shorter rollouts,

$\displaystyle K = 16, \qquad n = 5$

These are the values used in the original A3C work. Lunar Lander episodes are long and diverse, so many short segments from many episodes give a better mini-batch than a few long segments from fewer episodes.

We also use a smaller entropy bonus,

$\displaystyle \beta = 0.001$

With four actions and shaped rewards, there is little risk of premature collapse, while random engine firings are costly. We train for a larger budget of environment steps and evaluate less frequently. Finally, as in Chapter 08, we evaluate the **stochastic policy**, since on this environment the greedy policy can be much worse than the policy we optimise, especially early in training:

In [ ]:
lunar_lander = gymnasium.make('LunarLander-v3', render_mode="rgb_array")

print('State Space: ', lunar_lander.observation_space.shape)
print('Action Space: ', lunar_lander.action_space.n)

In [ ]:
# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

lunar_lander.reset(seed=seed)
lunar_lander.action_space.seed(seed)

# fresh networks and optimizers for the new environment
pi = create_policy_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
v = create_value_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=actor_learning_rate)
value_optimizer = torch.optim.RMSprop(v.parameters(), lr=critic_learning_rate)

random_score = evaluate(lunar_lander, lambda state: lunar_lander.action_space.sample(), episodes=100)
print("Random score: ", random_score)

In [ ]:
scores = a2c(lunar_lander, max_steps=1_000_000, eval_every=50_000, n_envs=16, n_steps=5, gamma=0.99, beta=0.001, lam=0.95, eval_policy=stochastic_pi)

In [ ]:
eval_every = 50_000

plt.figure(figsize=(12, 6))
plt.title("A2C with GAE on Lunar Lander (stochastic policy)")
plt.xlabel("Environment steps")
plt.ylabel("Score")
plt.plot((np.arange(len(scores)) + 1) * eval_every, scores, color="green", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", linewidth=1, label="Solved")
plt.legend()
plt.show()

In [ ]:
stochastic_score = evaluate(lunar_lander, stochastic_pi, episodes=10)
greedy_score = evaluate(lunar_lander, greedy_pi, episodes=10)

print("Random score", random_score)
print("Agent score (stochastic policy)", stochastic_score)
print("Agent score (greedy policy)", greedy_score)

In [ ]:
show_policy(lunar_lander, stochastic_pi, n_episodes=1, max_steps=1000)

The comparison with Chapter 08 is informative even though the horizontal axes differ. The VPG run used

$\displaystyle 3000$

episodes, which correspond to roughly half a million to a million environment steps, depending on how long the agent keeps the lander in the air. After that budget the Monte Carlo agent had reached scores around zero.

The actor-critic agent gets there much sooner. It stops crashing within the first hundred thousand steps, then learns to hover and drift towards the pad. Scores around zero, and occasionally positive scores, appear within the first few hundred thousand steps.

The reason is exactly the one developed in this chapter. Parallel environments decorrelate the mini-batches, and bootstrapped, lambda-weighted advantages are a much less noisy learning signal than returns summed over hundreds of rewards.

The learning curve is still far from monotonic. The score can oscillate by more than a hundred points from one evaluation to the next, and the agent can lose a good policy after having found it. A single run therefore rarely ends near the solved threshold,

$\displaystyle 200$

This is the **step-size problem** anticipated in Chapter 08. The actor is updated with a fixed learning rate, and with the large, noisy advantages of Lunar Lander, some updates move the policy too far. Gradient clipping limits the damage but does not remove the cause. Making policy updates safe is exactly what the trust-region methods of Chapter 11 do, and PPO will solve this environment reliably. The exercises invite you to run the multi-seed experiment and to vary the number of environments, the rollout length, the GAE parameter, and the learning rates.

## Limitations and Outlook

Actor-critic methods are the **template** of modern deep reinforcement learning: an actor that proposes actions, a critic that evaluates them, bootstrapped targets to keep the variance under control, parallel or batched experience to keep the samples diverse, and an entropy bonus to keep exploring. Every algorithm of the following chapters is an actor-critic method in this sense. The versions seen in this chapter, however, leave three problems open.

First, they remain **on-policy** and therefore **sample inefficient**. Every transition is used for exactly one gradient step. Parallel actors make the samples better, not cheaper: A2C still needs hundreds of thousands of environment steps on Lunar Lander, and each of them is seen exactly once, whereas a DQN-style agent replays every transition many times. Chapter 10 combines the actor-critic structure with the off-policy machinery of Chapter 07, for continuous actions, in the DDPG family of algorithms.

Second, the actor's **step size** is still a delicate hyperparameter, and a single large update can collapse the policy irreversibly. Gradient clipping limits the damage but does not address the cause, which is that a fixed step in parameter space can correspond to a very different change in the policy from one state to another. Chapter 11 introduces trust region methods, and PPO in particular, which constrain how much the policy may change at each update and make it safe to take **several gradient steps on the same rollout**, recovering part of the lost sample efficiency.

Third, the critic introduces **bias**, and bias does not average out with more data. GAE lets us trade bias for variance, but a critic that is systematically wrong in some region of the state space will systematically mislead the actor there, and the actor has no way of knowing. This is the price of bootstrapping, which we accepted in Chapter 05 for the same reasons, and it is why the quality of the critic, its architecture, learning rate and targets, receives so much attention in the algorithms that follow.